# Scaling Up: Polars, DuckDB, Dask and PySpark

Notebook for **Chapter 22** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch22.html

Tip: change a number, predict the output, then run the cell again.

## 22.1 What these libraries are, and when to leave pandas

In [ ]:
import importlib.util
from importlib.metadata import version, PackageNotFoundError

for name in ["pandas", "numpy", "pyarrow", "polars", "duckdb", "dask", "pyspark"]:
    if importlib.util.find_spec(name) is None:
        print(f"{name:8s} not installed")
    else:
        try:
            print(f"{name:8s} {version(name)}")
        except PackageNotFoundError:
            print(f"{name:8s} present")
import sqlite3
print("sqlite3  ", sqlite3.sqlite_version)

pandas   3.0.6
numpy    2.5.3
pyarrow  not installed
polars   not installed
duckdb   not installed
dask     not installed
pyspark  not installed
sqlite3   3.50.4


In [ ]:
# NOT RUN HERE (libraries not installed)
import pandas as pd
import polars as pl
import duckdb
import dask.dataframe as dd
import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql import SparkSession, functions as F, Window

In [ ]:
import io
import pandas as pd

orders = pd.read_csv(io.StringIO("""order_id,customer_id,region,category,amount
1,c1,EU,book,20
2,c2,EU,toy,35
3,c1,EU,toy,15
4,c3,US,book,50
5,c4,US,toy,10
6,c3,US,toy,25
7,c5,APAC,book,40
8,c5,APAC,book,30
9,c6,APAC,toy,5
10,c2,EU,book,45"""))
tiers = pd.DataFrame({"customer_id": ["c1", "c2", "c3", "c4", "c5", "c6"],
                      "tier": ["gold", "silver", "gold", "bronze", "silver", "bronze"]})

rev = orders.groupby(["region", "customer_id"], as_index=False)["amount"].sum()
rev = rev.rename(columns={"amount": "revenue"})
rev = rev.merge(tiers, on="customer_id", how="left")
rev["rank"] = rev.groupby("region")["revenue"].rank(method="first", ascending=False).astype(int)
print(rev.sort_values(["region", "rank"]).to_string(index=False))

region customer_id  revenue   tier  rank
  APAC          c5       70 silver     1
  APAC          c6        5 bronze     2
    EU          c2       80 silver     1
    EU          c1       35   gold     2
    US          c3       75   gold     1
    US          c4       10 bronze     2


In [ ]:
import io, sqlite3
import pandas as pd

csv = """order_id,customer_id,region,category,amount
1,c1,EU,book,20
2,c2,EU,toy,35
3,c1,EU,toy,15
4,c3,US,book,50
5,c4,US,toy,10
6,c3,US,toy,25
7,c5,APAC,book,40
8,c5,APAC,book,30
9,c6,APAC,toy,5
10,c2,EU,book,45"""
con = sqlite3.connect(":memory:")
pd.read_csv(io.StringIO(csv)).to_sql("orders", con, index=False)
con.execute("CREATE TABLE tiers(customer_id TEXT, tier TEXT)")
con.executemany("INSERT INTO tiers VALUES (?, ?)", [("c1","gold"),("c2","silver"),("c3","gold"),("c4","bronze"),("c5","silver"),("c6","bronze")])

sql = """
WITH rev AS (
    SELECT region, customer_id, SUM(amount) AS revenue
    FROM orders GROUP BY region, customer_id
)
SELECT r.region, r.customer_id, r.revenue, t.tier,
       ROW_NUMBER() OVER (PARTITION BY r.region ORDER BY r.revenue DESC) AS rnk
FROM rev r LEFT JOIN tiers t ON t.customer_id = r.customer_id
ORDER BY r.region, rnk
"""
for row in con.execute(sql):
    print(row)

('APAC', 'c5', 70, 'silver', 1)
('APAC', 'c6', 5, 'bronze', 2)
('EU', 'c2', 80, 'silver', 1)
('EU', 'c1', 35, 'gold', 2)
('US', 'c3', 75, 'gold', 1)
('US', 'c4', 10, 'bronze', 2)


## 22.2 Before you scale: measure, shrink and chunk in pandas

The cheapest speed-up is the one that needs no new library. Most "big data" problems at the 1 to 20 GB scale are really "wasteful data" problems: columns you never use, 64-bit numbers holding values below 100, and the word `"EU"` stored a million times. This section gives you the measuring tools and the three fixes: read less, store smaller, read in pieces.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1_000_000
df = pd.DataFrame({
    "order_id": np.arange(n),
    "customer_id": rng.integers(0, 50_000, n),
    "region": rng.choice(["EU", "US", "APAC", "LATAM", "MEA"], n),
    "qty": rng.integers(1, 20, n),
    "price": rng.normal(50, 15, n).round(2),
})
mb = lambda s: round(s.memory_usage(deep=True, index=False) / 1e6, 1)
print("BEFORE")
for c in df:
    print(f"  {c:12s} {str(df[c].dtype):6s} {mb(df[c]):6.1f} MB")
print("  total", round(df.memory_usage(deep=True).sum() / 1e6, 1), "MB")

small = df.assign(
    order_id=pd.to_numeric(df.order_id, downcast="unsigned"),
    customer_id=pd.to_numeric(df.customer_id, downcast="unsigned"),
    region=df.region.astype("category"),
    qty=pd.to_numeric(df.qty, downcast="unsigned"),
    price=df.price.astype("float32"),
)
print("AFTER")
for c in small:
    print(f"  {c:12s} {str(small[c].dtype):8s} {mb(small[c]):5.1f} MB")
print("  total", round(small.memory_usage(deep=True).sum() / 1e6, 1), "MB")

BEFORE
  order_id     int64     8.0 MB
  customer_id  int64     8.0 MB
  region       str      52.2 MB
  qty          int64     8.0 MB
  price        float64    8.0 MB
  total 84.2 MB
AFTER
  order_id     uint32     4.0 MB
  customer_id  uint16     2.0 MB
  region       category   1.0 MB
  qty          uint8      1.0 MB
  price        float32    4.0 MB
  total 12.0 MB


In [ ]:
import os, tempfile, tracemalloc
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1_000_000
df = pd.DataFrame({
    "order_id": np.arange(n),
    "customer_id": rng.integers(0, 50_000, n),
    "region": rng.choice(["EU", "US", "APAC", "LATAM", "MEA"], n),
    "qty": rng.integers(1, 20, n),
    "price": rng.normal(50, 15, n).round(2),
})
path = os.path.join(tempfile.mkdtemp(), "orders.csv")
df.to_csv(path, index=False)
del df
print("file size over 20 MB:", os.path.getsize(path) > 20e6)

def revenue_full():
    d = pd.read_csv(path)
    return (d.qty * d.price).groupby(d.region).sum()

def revenue_chunked(size):
    total = None
    for ch in pd.read_csv(path, chunksize=size):
        part = (ch.qty * ch.price).groupby(ch.region).sum()
        total = part if total is None else total.add(part, fill_value=0)
    return total

peaks, answers = {}, {}
for label, fn in [("full", revenue_full),
                  ("chunk 100k", lambda: revenue_chunked(100_000)),
                  ("chunk 10k", lambda: revenue_chunked(10_000))]:
    tracemalloc.start()
    answers[label] = fn().sort_index()
    peaks[label] = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

print("full peak above 60 MB:", peaks["full"] > 60e6)
print("chunk 100k peak below 30 MB:", peaks["chunk 100k"] < 30e6)
print("chunk 10k peak below 10 MB:", peaks["chunk 10k"] < 10e6)
print("same answer:", np.allclose(answers["full"], answers["chunk 10k"]))
print(answers["chunk 10k"].round(0).astype("int64").to_string())

file size over 20 MB: True
full peak above 60 MB: True
chunk 100k peak below 30 MB: True
chunk 10k peak below 10 MB: True
same answer: True
region
APAC     100744849
EU        99851750
LATAM     99823225
MEA       99991196
US        99891792


In [ ]:
import os, tempfile, time
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1_000_000
df = pd.DataFrame({
    "order_id": np.arange(n),
    "customer_id": rng.integers(0, 50_000, n),
    "region": rng.choice(["EU", "US", "APAC", "LATAM", "MEA"], n),
    "qty": rng.integers(1, 20, n),
    "price": rng.normal(50, 15, n).round(2),
})
path = os.path.join(tempfile.mkdtemp(), "orders.csv")
df.to_csv(path, index=False)

def best(fn, reps=5):
    times = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return min(times)

print("--- reading the CSV (best of 5, seconds) ---")
print("read everything       ", round(best(lambda: pd.read_csv(path)), 3))
print("usecols 2 of 5        ", round(best(lambda: pd.read_csv(path, usecols=["region", "price"])), 3))
print("usecols + dtype       ", round(best(lambda: pd.read_csv(path, usecols=["region", "price"], dtype={"region": "category", "price": "float32"})), 3))
print("chunks of 100k, sum   ", round(best(lambda: sum(c.price.sum() for c in pd.read_csv(path, chunksize=100_000))), 3))

df_cat = df.assign(region=df.region.astype("category"))
codes = df_cat.region.cat.codes.to_numpy()
price = df.price.to_numpy()
region = df.region.tolist()
def py_loop():
    tot = {}
    for r, p in zip(region, price):
        tot[r] = tot.get(r, 0.0) + p
    return tot
print("--- mean price per region (best of 5, seconds) ---")
print("python dict loop      ", round(best(py_loop), 3))
print("groupby str column    ", round(best(lambda: df.groupby("region")["price"].mean()), 3))
print("groupby category      ", round(best(lambda: df_cat.groupby("region", observed=True)["price"].mean()), 3))
print("numpy bincount codes  ", round(best(lambda: np.bincount(codes, weights=price) / np.bincount(codes)), 3))

## 22.3 Parquet and columnar file formats

In [ ]:
import os, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1_000_000
df = pd.DataFrame({
    "order_id": np.arange(n, dtype="uint32"),
    "customer_id": rng.integers(0, 50_000, n).astype("uint16"),
    "region": pd.Categorical(rng.choice(["EU", "US", "APAC", "LATAM", "MEA"], n)),
    "qty": rng.integers(1, 20, n).astype("uint8"),
    "price": rng.normal(50, 15, n).round(2),
})
d = tempfile.mkdtemp()
csv_path = os.path.join(d, "orders.csv")
df.to_csv(csv_path, index=False)
for c in ["order_id", "customer_id", "qty", "price"]:
    np.save(os.path.join(d, c + ".npy"), df[c].to_numpy())
np.save(os.path.join(d, "region_codes.npy"), df["region"].cat.codes.to_numpy())

size = lambda p: os.path.getsize(p)
col_files = [f for f in os.listdir(d) if f.endswith(".npy")]
print("csv MB          :", round(size(csv_path) / 1e6, 1))
print("all columns MB  :", round(sum(size(os.path.join(d, f)) for f in col_files) / 1e6, 1))
print("price.npy MB    :", round(size(os.path.join(d, "price.npy")) / 1e6, 1))

total_csv = pd.read_csv(csv_path)["price"].sum()
price = np.load(os.path.join(d, "price.npy"), mmap_mode="r")
total_npy = float(price.sum())
print("same total      :", np.isclose(total_csv, total_npy))
print("bytes touched, binary vs csv:", f"{size(os.path.join(d, 'price.npy')) / size(csv_path):.0%}")

csv MB          : 25.3
all columns MB  : 16.0
price.npy MB    : 8.0
same total      : True
bytes touched, binary vs csv: 32%


In [ ]:
import os, tempfile
from pathlib import Path
import numpy as np
import pandas as pd

rng = np.random.default_rng(5)
n = 100_000
df = pd.DataFrame({
    "year": rng.choice([2024, 2025], n),
    "region": rng.choice(["EU", "US", "APAC"], n),
    "amount": rng.integers(1, 500, n),
})
root = Path(tempfile.mkdtemp()) / "sales"
for (year, region), part in df.groupby(["year", "region"]):
    folder = root / f"year={year}" / f"region={region}"
    folder.mkdir(parents=True)
    part[["amount"]].to_csv(folder / "part-0.csv", index=False)

for p in sorted(root.rglob("*.csv")):
    print(p.relative_to(root))

wanted = root / "year=2025" / "region=EU" / "part-0.csv"
fast = pd.read_csv(wanted)
print("rows read with pruning:", len(fast), "of", n)
full = df[(df.year == 2025) & (df.region == "EU")]["amount"].sum()
print("same total:", fast["amount"].sum() == full)

year=2024/region=APAC/part-0.csv
year=2024/region=EU/part-0.csv
year=2024/region=US/part-0.csv
year=2025/region=APAC/part-0.csv
year=2025/region=EU/part-0.csv
year=2025/region=US/part-0.csv
rows read with pruning: 16521 of 100000
same total: True


In [ ]:
# NOT RUN HERE (library not installed)
import pandas as pd
import duckdb
import polars as pl
import pyarrow.dataset as ds

# df is the 100,000-row DataFrame (year, region, amount) from the block above
# 1. pandas + pyarrow: write a partitioned folder, then read with a filter
df.to_parquet("sales", partition_cols=["year", "region"], index=False)
eu_2025 = pd.read_parquet("sales", filters=[("year", "==", 2025), ("region", "==", "EU")])

# 2. DuckDB: the folder names become columns; the WHERE clause prunes folders
total = duckdb.sql("""
    SELECT SUM(amount) FROM read_parquet('sales/*/*/*.parquet', hive_partitioning = true)
    WHERE year = 2025 AND region = 'EU'
""").fetchone()[0]

# 3. Polars lazy: scan, filter, aggregate, collect
total = (pl.scan_parquet("sales/**/*.parquet", hive_partitioning=True)
           .filter((pl.col("year") == 2025) & (pl.col("region") == "EU"))
           .select(pl.col("amount").sum())
           .collect()
           .item())

# 4. pyarrow dataset
dataset = ds.dataset("sales", format="parquet", partitioning="hive")
table = dataset.to_table(filter=(ds.field("year") == 2025) & (ds.field("region") == "EU"), columns=["amount"])

## 22.4 Polars: fast DataFrames with expressions and a lazy engine

In [ ]:
# NOT RUN HERE (library not installed)
import polars as pl

df = pl.DataFrame({
    "region": ["EU", "EU", "US", "APAC"],
    "amount": [20, 35, 50, 40],
})
print(df.shape)                                   # (4, 2)
print(df.schema)                                  # region: String, amount: Int64
out = df.with_columns(
    amount_with_vat=pl.col("amount") * 1.2,
    big=pl.col("amount") > 30,
)
print(out)
print(df.select(pl.col("amount").sum()).item())   # 145

In [ ]:
# NOT RUN HERE (library not installed)
import polars as pl

orders = pl.read_csv("orders.csv")
tiers = pl.DataFrame({
    "customer_id": ["c1", "c2", "c3", "c4", "c5", "c6"],
    "tier": ["gold", "silver", "gold", "bronze", "silver", "bronze"],
})

result = (
    orders
    .group_by(["region", "customer_id"], maintain_order=True)
    .agg(pl.col("amount").sum().alias("revenue"))
    .join(tiers, on="customer_id", how="left")
    .with_columns(
        pl.col("revenue").rank("ordinal", descending=True).over("region").alias("rank")
    )
    .sort(["region", "rank"])
)
print(result)

In [ ]:
# NOT RUN HERE (library not installed)
import polars as pl

lf = (
    pl.scan_csv("events.csv", try_parse_dates=True)
    .filter(pl.col("country") == "DE")
    .group_by("user_id")
    .agg(
        pl.len().alias("events"),
        pl.col("price").sum().alias("spend"),
    )
    .filter(pl.col("events") >= 5)
    .sort("spend", descending=True)
)

print(lf.explain())                               # read the optimized plan first
top = lf.head(1000).collect()                     # run it, keep the top 1000 only
lf.sink_parquet("german_users.parquet")           # or stream the whole result to disk

In [ ]:
import io, os, tempfile
import pandas as pd

csv = """order_id,customer_id,region,category,amount
1,c1,EU,book,20
2,c2,EU,toy,35
3,c1,EU,toy,15
4,c3,US,book,50
5,c4,US,toy,10
6,c3,US,toy,25
7,c5,APAC,book,40
8,c5,APAC,book,30
9,c6,APAC,toy,5
10,c2,EU,book,45
"""
path = os.path.join(tempfile.mkdtemp(), "orders.csv")
open(path, "w").write(csv)

class Lazy:
    def __init__(self, path):
        self.path, self.steps = path, []
    def filter(self, col, test):
        self.steps.append(("filter", col, test)); return self
    def select(self, *cols):
        self.steps.append(("select", cols)); return self
    def collect(self, optimize=True):
        needed = set()
        for s in self.steps:
            needed.update([s[1]] if s[0] == "filter" else s[1])
        usecols = sorted(needed) if optimize else None
        df = pd.read_csv(self.path, usecols=usecols)
        print("columns parsed:", list(df.columns))
        out = list(df.columns)
        for s in self.steps:
            if s[0] == "filter":
                df = df[s[2](df[s[1]])]
            else:
                out = list(s[1])
        return df[out]

def plan():
    return Lazy(path).filter("region", lambda s: s == "EU").select("customer_id", "amount")

print("naive:")
a = plan().collect(optimize=False)
print("optimized:")
b = plan().collect()
print("same result:", a.equals(b))
print(b.to_string(index=False))

naive:
columns parsed: ['order_id', 'customer_id', 'region', 'category', 'amount']
optimized:
columns parsed: ['customer_id', 'region', 'amount']
same result: True
customer_id  amount
         c1      20
         c2      35
         c1      15
         c2      45


## 22.5 DuckDB: SQL straight on files and DataFrames

In [ ]:
# NOT RUN HERE (library not installed)
import duckdb
import pandas as pd

print(duckdb.__version__)

# 1. Query a CSV file by name. No loading step.
print(duckdb.sql("SELECT region, SUM(amount) AS revenue FROM 'orders.csv' GROUP BY region ORDER BY region").fetchall())

# 2. Query a pandas DataFrame by its variable name (a "replacement scan").
df = pd.read_csv("orders.csv")
print(duckdb.sql("SELECT region, SUM(amount) AS revenue FROM df GROUP BY region ORDER BY region").df())

In [ ]:
import io, sqlite3
import pandas as pd

csv = """order_id,customer_id,region,category,amount
1,c1,EU,book,20
2,c2,EU,toy,35
3,c1,EU,toy,15
4,c3,US,book,50
5,c4,US,toy,10
6,c3,US,toy,25
7,c5,APAC,book,40
8,c5,APAC,book,30
9,c6,APAC,toy,5
10,c2,EU,book,45"""
df = pd.read_csv(io.StringIO(csv))
con = sqlite3.connect(":memory:")
df.to_sql("df", con, index=False)
rows = con.execute("SELECT region, SUM(amount) AS revenue FROM df GROUP BY region ORDER BY region").fetchall()
print(rows)
print(pd.read_sql("SELECT category, COUNT(*) AS n, AVG(amount) AS avg_amount FROM df GROUP BY category ORDER BY category", con))

[('APAC', 75), ('EU', 115), ('US', 85)]
  category  n  avg_amount
0     book  5        37.0
1      toy  5        18.0


In [ ]:
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE sales(day INTEGER, region TEXT, amount INTEGER)")
con.executemany("INSERT INTO sales VALUES (?, ?, ?)", [
    (1, "EU", 10), (2, "EU", 30), (3, "EU", 30), (4, "EU", 20),
    (1, "US", 5), (2, "US", 15), (3, "US", 10),
])
q = """
SELECT region, day, amount,
  SUM(amount) OVER (PARTITION BY region ORDER BY day)                  AS running,
  amount - LAG(amount) OVER (PARTITION BY region ORDER BY day)         AS change,
  AVG(amount) OVER (PARTITION BY region ORDER BY day
                    ROWS BETWEEN 1 PRECEDING AND CURRENT ROW)          AS avg2,
  RANK()       OVER (PARTITION BY region ORDER BY amount DESC)         AS rnk,
  DENSE_RANK() OVER (PARTITION BY region ORDER BY amount DESC)         AS drnk
FROM sales ORDER BY region, day
"""
print("region day amount running change avg2 rnk drnk")
for r in con.execute(q):
    print(*r)

region day amount running change avg2 rnk drnk
EU 1 10 10 None 10.0 4 3
EU 2 30 40 20 20.0 1 1
EU 3 30 70 0 30.0 1 1
EU 4 20 90 -10 25.0 3 2
US 1 5 5 None 5.0 3 3
US 2 15 20 10 10.0 1 1
US 3 10 30 -5 12.5 2 2


In [ ]:
# NOT RUN HERE (library not installed)
import duckdb

duckdb.sql("""
    CREATE OR REPLACE TABLE rev AS
    SELECT o.region, o.customer_id, SUM(o.amount) AS revenue
    FROM 'orders.csv' o
    GROUP BY o.region, o.customer_id
""")

tiers = duckdb.sql("""
    SELECT * FROM (VALUES ('c1','gold'),('c2','silver'),('c3','gold'),
                          ('c4','bronze'),('c5','silver'),('c6','bronze')) AS t(customer_id, tier)
""")
duckdb.register("tiers", tiers)

# All ranks, same SQL text as the SQLite block in Section 22.1:
print(duckdb.sql("""
    SELECT r.region, r.customer_id, r.revenue, t.tier,
           ROW_NUMBER() OVER (PARTITION BY r.region ORDER BY r.revenue DESC) AS rnk
    FROM rev r LEFT JOIN tiers t USING (customer_id)
    ORDER BY r.region, rnk
""").fetchall())

# Only the top customer per region, using QUALIFY (DuckDB; SQLite does not have it):
print(duckdb.sql("""
    SELECT r.region, r.customer_id, r.revenue
    FROM rev r
    QUALIFY ROW_NUMBER() OVER (PARTITION BY r.region ORDER BY r.revenue DESC) = 1
    ORDER BY r.region
""").fetchall())

In [ ]:
import io, sqlite3
import pandas as pd

csv = """order_id,customer_id,region,category,amount
1,c1,EU,book,20
2,c2,EU,toy,35
3,c1,EU,toy,15
4,c3,US,book,50
5,c4,US,toy,10
6,c3,US,toy,25
7,c5,APAC,book,40
8,c5,APAC,book,30
9,c6,APAC,toy,5
10,c2,EU,book,45"""
con = sqlite3.connect(":memory:")
pd.read_csv(io.StringIO(csv)).to_sql("orders", con, index=False)
q = """
SELECT region, customer_id, revenue FROM (
    SELECT region, customer_id, SUM(amount) AS revenue,
           ROW_NUMBER() OVER (PARTITION BY region ORDER BY SUM(amount) DESC) AS rn
    FROM orders GROUP BY region, customer_id
) WHERE rn = 1 ORDER BY region
"""
print(con.execute(q).fetchall())

[('APAC', 'c5', 70), ('EU', 'c2', 80), ('US', 'c3', 75)]


In [ ]:
# NOT RUN HERE (library not installed)
import duckdb

con = duckdb.connect("analytics.duckdb")          # a persistent database file
con.execute("SET memory_limit = '6GB'")           # cap RAM; bigger operators spill to disk
con.execute("SET temp_directory = '/tmp/duck_spill'")
con.execute("SET threads = 4")

# Convert a huge CSV into partitioned Parquet in one streaming query.
con.execute("""
    COPY (
        SELECT *, year(CAST(ts AS TIMESTAMP)) AS year
        FROM read_csv('events.csv', header = true)
    ) TO 'events_pq' (FORMAT PARQUET, PARTITION_BY (year), COMPRESSION zstd)
""")

# Query it back; the WHERE prunes the year folders.
top = con.execute("""
    SELECT user_id, COUNT(*) AS n
    FROM read_parquet('events_pq/*/*.parquet', hive_partitioning = true)
    WHERE year = 2025
    GROUP BY user_id
    ORDER BY n DESC
    LIMIT 10
""").df()
print(top)
con.execute("EXPLAIN ANALYZE SELECT COUNT(*) FROM read_parquet('events_pq/*/*.parquet')").fetchall()

## 22.6 Dask: pandas in pieces, with a scheduler

In [ ]:
# NOT RUN HERE (library not installed)
import dask.dataframe as dd

ddf = dd.read_csv("orders.csv")
print(ddf.npartitions)                       # 1 for a tiny file; many for a big one
lazy = ddf.groupby("region").amount.sum()    # nothing has been computed yet
print(lazy.compute().sort_index())           # APAC 75, EU 115, US 85

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
n = 300_000
df = pd.DataFrame({"region": rng.choice(["EU", "US", "APAC"], n), "price": rng.normal(50, 15, n)})

cuts = [0, 20_000, 120_000, n]                          # unequal partitions on purpose
parts = [df.iloc[a:b] for a, b in zip(cuts[:-1], cuts[1:])]
print("partition sizes:", [len(p) for p in parts])

partials = [p.groupby("region")["price"].agg(["sum", "count"]) for p in parts]
combined = sum(partials)                                 # element-wise add, aligned on region
mean_correct = combined["sum"] / combined["count"]

mean_of_means = pd.concat([p.groupby("region")["price"].mean() for p in parts], axis=1).mean(axis=1)
truth = df.groupby("region")["price"].mean()

print("sum/count equals pandas:", np.allclose(mean_correct.sort_index(), truth.sort_index()))
print("mean of means equals pandas:", np.allclose(mean_of_means.sort_index(), truth.sort_index()))
print("worst error of mean-of-means:", float((mean_of_means - truth).abs().max()) > 1e-6)

partition sizes: [20000, 100000, 180000]
sum/count equals pandas: True
mean of means equals pandas: False
worst error of mean-of-means: True


In [ ]:
# NOT RUN HERE (library not installed)
import dask.dataframe as dd
import pandas as pd

orders = dd.read_csv("orders.csv")
tiers = pd.DataFrame({"customer_id": ["c1", "c2", "c3", "c4", "c5", "c6"],
                      "tier": ["gold", "silver", "gold", "bronze", "silver", "bronze"]})

rev = (orders.groupby(["region", "customer_id"]).amount.sum()
             .reset_index().rename(columns={"amount": "revenue"}))
rev = rev.merge(tiers, on="customer_id", how="left")     # small pandas side: cheap join

def add_rank(part):
    part = part.copy()
    part["rank"] = part.groupby("region")["revenue"].rank(method="first", ascending=False).astype(int)
    return part

ranked = rev.shuffle(on="region").map_partitions(add_rank, meta=rev._meta.assign(rank=0))
print(ranked.compute().sort_values(["region", "rank"]))

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2)
orders = pd.DataFrame({"customer_id": rng.integers(0, 5_000, 100_000), "amount": rng.integers(1, 100, 100_000)})
customers = pd.DataFrame({"customer_id": np.arange(5_000), "tier": rng.choice(["gold", "silver", "bronze"], 5_000)})
K = 4

def split(frame):
    bucket = pd.util.hash_pandas_object(frame["customer_id"], index=False) % K
    return [frame[bucket == i] for i in range(K)]

o_parts, c_parts = split(orders), split(customers)
print("order rows per partition:", [len(p) for p in o_parts])
joined = pd.concat([o.merge(c, on="customer_id", how="left") for o, c in zip(o_parts, c_parts)])
direct = orders.merge(customers, on="customer_id", how="left")
key = ["customer_id", "amount", "tier"]
print("rows:", len(joined), len(direct))
print("same result:", joined.sort_values(key).reset_index(drop=True).equals(direct.sort_values(key).reset_index(drop=True)))

order rows per partition: [23741, 26181, 24196, 25882]
rows: 100000 100000
same result: True


## 22.7 PySpark: partitions, shuffles, joins, skew, caching and the Spark UI

In [ ]:
# NOT RUN HERE (library not installed)
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.appName("ch22").master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")      # the default is 200, too many for tiny data
         .getOrCreate())

orders = spark.read.csv("orders.csv", header=True, inferSchema=True)
tiers = spark.createDataFrame(
    [("c1", "gold"), ("c2", "silver"), ("c3", "gold"),
     ("c4", "bronze"), ("c5", "silver"), ("c6", "bronze")],
    ["customer_id", "tier"],
)

rev = orders.groupBy("region", "customer_id").agg(F.sum("amount").alias("revenue"))
w = Window.partitionBy("region").orderBy(F.col("revenue").desc())
result = (rev.join(tiers, "customer_id", "left")
             .withColumn("rank", F.row_number().over(w))
             .select("region", "customer_id", "revenue", "tier", "rank")
             .orderBy("region", "rank"))
result.show()
result.explain()
spark.stop()

In [ ]:
# NOT RUN HERE (library not installed)
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").getOrCreate()
events = spark.read.parquet("events_pq")          # big
users = spark.read.parquet("users_pq")            # small (a few MB)

slim = events.select("user_id", "price").filter(F.col("price") > 0)   # prune early
joined = slim.join(F.broadcast(users), "user_id", "left")               # no shuffle of events
joined.explain()                                  # expect BroadcastHashJoin, no Exchange on events

clean = joined.cache()                            # lazy: nothing happens yet
print(clean.count())                              # first action: computes and fills the cache
print(clean.groupBy("country").agg(F.sum("price")).count())   # reuses the cache
clean.unpersist()

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n, n_keys, n_parts = 1_000_000, 1000, 8
w = 1 / np.arange(1, n_keys + 1)
keys = rng.choice(n_keys, n, p=w / w.sum())
vals = rng.integers(1, 100, n)

def part_sizes(k):
    h = pd.util.hash_array(k.astype(np.int64))
    return np.bincount(h % n_parts, minlength=n_parts)

before = part_sizes(keys)
print("hottest key share:", round((keys == 0).mean(), 3))
print("before:", before.tolist(), "max/mean =", round(before.max() / before.mean(), 2))
salt = rng.integers(0, 8, n)
after = part_sizes(keys * 8 + salt)
print("after :", after.tolist(), "max/mean =", round(after.max() / after.mean(), 2))

df = pd.DataFrame({"k": keys, "salt": salt, "v": vals})
stage1 = df.groupby(["k", "salt"])["v"].sum().reset_index()
stage2 = stage1.groupby("k")["v"].sum()
direct = df.groupby("k")["v"].sum()
print("same answer:", stage2.equals(direct))

hottest key share: 0.134
before: [247565, 101396, 117133, 71796, 152006, 160663, 60940, 88501] max/mean = 1.98
after : [138513, 128210, 121237, 106290, 161283, 138354, 91594, 114519] max/mean = 1.29
same answer: True


In [ ]:
# NOT RUN HERE (library not installed)
from pyspark.sql import functions as F

SALT = 8
# Two-stage aggregation for a skewed group-by key
stage1 = (df.withColumn("salt", (F.rand(seed=1) * SALT).cast("int"))
            .groupBy("k", "salt").agg(F.sum("v").alias("s")))
total = stage1.groupBy("k").agg(F.sum("s").alias("v"))

# Skewed join: salt the big side, replicate the small side across all salts
big = big.withColumn("salt", (F.rand(seed=1) * SALT).cast("int"))
small = small.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT)])))
joined = big.join(small, ["k", "salt"])

# Often enough: let Spark do it
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

In [ ]:
import time
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 200_000
df = pd.DataFrame({"amount": rng.integers(1, 500, n), "region": rng.choice(["EU", "US", "APAC"], n)})
rate = {"EU": 0.92, "US": 1.0, "APAC": 0.0065}

def row_udf(row):                      # like a plain Spark UDF: called once per row
    return row["amount"] * rate[row["region"]]

def best(fn, reps=3):
    times = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return min(times)

slow = best(lambda: df.apply(row_udf, axis=1))
fast = best(lambda: df["amount"] * df["region"].map(rate))      # column-at-a-time, like F.when / F.col arithmetic
print("row-at-a-time:", round(slow, 3), "s; vectorized:", round(fast, 4), "s; ratio:", round(slow / fast))
print("same values:", np.allclose(df.apply(row_udf, axis=1), df["amount"] * df["region"].map(rate)))

In [ ]:
# NOT RUN HERE (library not installed)
from pyspark.sql import functions as F
import pandas as pd

rates = {"EU": 0.92, "US": 1.0, "APAC": 0.0065}

# 1. Built-in: a map literal and column arithmetic (best)
rate_col = F.create_map([F.lit(x) for kv in rates.items() for x in kv])
df = df.withColumn("eur", F.col("amount") * rate_col[F.col("region")])

# 2. Vectorized pandas UDF: batches of rows as pandas Series
@F.pandas_udf("double")
def to_eur(amount: pd.Series, region: pd.Series) -> pd.Series:
    return amount * region.map(rates)
df = df.withColumn("eur2", to_eur("amount", "region"))

# 3. Plain UDF: one row at a time (slowest)
to_eur_slow = F.udf(lambda a, r: float(a * rates[r]), "double")
df = df.withColumn("eur3", to_eur_slow("amount", "region"))

## 22.8 How to choose

Start with the size of the data and the size of your machine, then ask what you already know. The flow chart below encodes the order of questions. Follow it from the top; stop at the first tool that answers yes.

In [ ]:
import os, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
n = 200_000
df = pd.DataFrame({
    "order_id": np.arange(n),
    "region": rng.choice(["EU", "US", "APAC", "LATAM", "MEA"], n),
    "qty": rng.integers(1, 20, n),
    "price": rng.normal(50, 15, n).round(2),
})
path = os.path.join(tempfile.mkdtemp(), "o.csv")
df.to_csv(path, index=False)

total_rows = sum(1 for _ in open(path)) - 1
sample = pd.read_csv(path, nrows=20_000)
per_row = sample.memory_usage(deep=True, index=False).sum() / len(sample)
estimate = per_row * total_rows
actual = pd.read_csv(path).memory_usage(deep=True, index=False).sum()
print("rows:", total_rows)
print("estimate within 5% of actual:", abs(estimate - actual) / actual < 0.05)
print("in-memory is larger than the file:", actual > os.path.getsize(path))

rows: 200000
estimate within 5% of actual: True
in-memory is larger than the file: True


## 22.9 Recipes: How do I ...?

Every recipe below follows the same shape: the task, a runnable pandas or standard-library solution with a check, and the equivalent in the big libraries where it differs. All files are generated in temporary folders.

In [ ]:
import os, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(11)
n = 300_000
raw = pd.DataFrame({
    "region": rng.choice(["eu", "US", " apac", "EU"], n),
    "qty": rng.integers(-2, 10, n),
    "price": np.where(rng.random(n) < 0.05, np.nan, rng.normal(40, 10, n).round(2)),
})
path = os.path.join(tempfile.mkdtemp(), "dirty.csv")
raw.to_csv(path, index=False)

revenue, kept, dropped = {}, 0, 0
for chunk in pd.read_csv(path, chunksize=50_000):
    chunk["region"] = chunk["region"].str.strip().str.upper()
    ok = chunk["qty"].gt(0) & chunk["price"].notna()
    dropped += int((~ok).sum())
    good = chunk[ok]
    kept += len(good)
    for region, value in (good["qty"] * good["price"]).groupby(good["region"]).sum().items():
        revenue[region] = revenue.get(region, 0.0) + value

print("kept", kept, "dropped", dropped, "total", kept + dropped)
print({k: round(v) for k, v in sorted(revenue.items())})

whole = raw.assign(region=raw.region.str.strip().str.upper())
whole = whole[whole.qty.gt(0) & whole.price.notna()]
check = (whole.qty * whole.price).groupby(whole.region).sum()
print("matches one-shot pandas:", all(abs(check[k] - revenue[k]) < 1e-6 * check[k] for k in check.index))

kept 213790 dropped 86210 total 300000
{'APAC': 10740859, 'EU': 21399204, 'US': 10654963}
matches one-shot pandas: True


In [ ]:
import os, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(21)
n = 200_000
df = pd.DataFrame({
    "order_id": rng.integers(0, 120_000, n),          # many repeated ids
    "ts": np.arange(n),                                # later row = newer version
    "status": rng.choice(["new", "paid", "shipped"], n),
})
d = tempfile.mkdtemp()
src = os.path.join(d, "orders.csv")
df.to_csv(src, index=False)

B = 8
for chunk in pd.read_csv(src, chunksize=40_000):                       # pass 1: scatter
    bucket = pd.util.hash_pandas_object(chunk["order_id"], index=False) % B
    for b, part in chunk.groupby(bucket):
        part.to_csv(os.path.join(d, f"bucket{b}.csv"), mode="a", header=not os.path.exists(os.path.join(d, f"bucket{b}.csv")), index=False)

kept = []
for b in range(B):                                                      # pass 2: dedupe each bucket
    part = pd.read_csv(os.path.join(d, f"bucket{b}.csv"))
    kept.append(part.sort_values("ts").drop_duplicates("order_id", keep="last"))
result = pd.concat(kept)

truth = df.sort_values("ts").drop_duplicates("order_id", keep="last")
print("rows in:", n, "unique ids:", len(result), "expected:", len(truth))
print("same rows:", result.sort_values("order_id").reset_index(drop=True).equals(truth.sort_values("order_id").reset_index(drop=True)))
print("largest bucket rows:", max(len(k) for k in kept) < n / 2)

rows in: 200000 unique ids: 97410 expected: 97410
same rows: True
largest bucket rows: True


In [ ]:
import json, os, tempfile
import numpy as np
import pandas as pd

d = tempfile.mkdtemp()
state_path = os.path.join(d, "state.json")
rng = np.random.default_rng(8)
for day in (1, 2):
    pd.DataFrame({"region": rng.choice(["EU", "US"], 1000), "amount": rng.integers(1, 100, 1000)}).to_csv(os.path.join(d, f"day{day}.csv"), index=False)

def update(folder):
    state = json.load(open(state_path)) if os.path.exists(state_path) else {"files": [], "sum": {}, "count": {}}
    for name in sorted(f for f in os.listdir(folder) if f.startswith("day")):
        if name in state["files"]:
            continue                                          # already processed: idempotent
        part = pd.read_csv(os.path.join(folder, name)).groupby("region")["amount"].agg(["sum", "count"])
        for region, row in part.iterrows():
            state["sum"][region] = state["sum"].get(region, 0) + int(row["sum"])
            state["count"][region] = state["count"].get(region, 0) + int(row["count"])
        state["files"].append(name)
    json.dump(state, open(state_path, "w"))
    return state

s1 = update(d)
print("after run 1:", s1["files"], s1["count"])
s2 = update(d)
print("after rerun:", s2["files"], s2["count"] == s1["count"])
pd.DataFrame({"region": rng.choice(["EU", "US"], 500), "amount": rng.integers(1, 100, 500)}).to_csv(os.path.join(d, "day3.csv"), index=False)
s3 = update(d)
print("after day 3:", s3["files"], sum(s3["count"].values()))
print("mean EU:", round(s3["sum"]["EU"] / s3["count"]["EU"], 2))

after run 1: ['day1.csv', 'day2.csv'] {'EU': 1010, 'US': 990}
after rerun: ['day1.csv', 'day2.csv'] True
after day 3: ['day1.csv', 'day2.csv', 'day3.csv'] 2500
mean EU: 49.49


In [ ]:
import io, os, sqlite3, tempfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(4)
n = 200_000
big = pd.DataFrame({"customer_id": rng.integers(0, 1_000, n), "amount": rng.integers(1, 100, n)})
small = pd.DataFrame({"customer_id": np.arange(0, 900), "tier": rng.choice(["gold", "silver", "bronze"], 900)})   # 100 customers unknown
path = os.path.join(tempfile.mkdtemp(), "big.csv")
big.to_csv(path, index=False)

lookup = small.set_index("customer_id")["tier"]
totals, unmatched = {}, 0
for chunk in pd.read_csv(path, chunksize=50_000):
    tier = chunk["customer_id"].map(lookup)                    # hash lookup, no shuffle
    unmatched += int(tier.isna().sum())
    for t, v in chunk["amount"].groupby(tier.fillna("unknown")).sum().items():
        totals[t] = totals.get(t, 0) + int(v)
print(dict(sorted(totals.items())), "unmatched rows:", unmatched)

con = sqlite3.connect(":memory:")
big.to_sql("big", con, index=False); small.to_sql("small", con, index=False)
rows = con.execute("""SELECT COALESCE(s.tier, 'unknown') AS tier, SUM(b.amount)
                      FROM big b LEFT JOIN small s ON s.customer_id = b.customer_id
                      GROUP BY 1 ORDER BY 1""").fetchall()
print("sqlite3 agrees:", dict(rows) == totals)

{'bronze': 2996573, 'gold': 3034570, 'silver': 2974233, 'unknown': 1011012} unmatched rows: 20123
sqlite3 agrees: True


In [ ]:
import os, tempfile
from pathlib import Path
import numpy as np
import pandas as pd

rng = np.random.default_rng(6)
n = 120_000
src = os.path.join(tempfile.mkdtemp(), "events.csv")
pd.DataFrame({
    "ts": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "country": rng.choice(["DE", "FR", "US"], n),
    "value": rng.integers(1, 1000, n),
}).to_csv(src, index=False)

root = Path(tempfile.mkdtemp()) / "events"
rows_written = 0
for chunk in pd.read_csv(src, chunksize=30_000, parse_dates=["ts"]):
    chunk["month"] = chunk["ts"].dt.strftime("%Y-%m")
    for (month, country), part in chunk.groupby(["month", "country"]):
        folder = root / f"month={month}" / f"country={country}"
        folder.mkdir(parents=True, exist_ok=True)
        target = folder / "part-0.csv"
        part[["ts", "value"]].to_csv(target, mode="a", header=not target.exists(), index=False)
        rows_written += len(part)

files = sorted(root.rglob("*.csv"))
print("folders:", len({p.parent for p in files}), "rows written:", rows_written == n)
print(files[0].relative_to(root))
only = pd.read_csv(root / "month=2025-02" / "country=DE" / "part-0.csv")
print("one partition rows:", len(only), "of", n)

folders: 9 rows written: True
month=2025-01/country=DE/part-0.csv
one partition rows: 12355 of 120000


## 22.10 Performance and pitfalls

These are the mistakes that cost the most hours. Each box gives the wrong code, the symptom you will see, and the fix.

## 22.11 Cross-library integration

You will often mix tools: aggregate in DuckDB or Spark, then finish in pandas, then plot or model. The rule is to move *small* things between libraries and leave big things on disk as Parquet.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n = 200_000
X = rng.normal(size=(n, 3))
y = X @ np.array([3.0, -2.0, 0.5]) + 10 + rng.normal(0, 1, n)

full = LinearRegression().fit(X, y)

scaler = StandardScaler().fit(X[:20_000])                 # in practice: fit the scaler on one chunk or a sample
model = SGDRegressor(learning_rate="constant", eta0=0.001, random_state=0)
for epoch in range(3):
    for start in range(0, n, 20_000):
        sl = slice(start, start + 20_000)
        model.partial_fit(scaler.transform(X[sl]), y[sl])

pred_full = full.predict(X[:2000])
pred_sgd = model.predict(scaler.transform(X[:2000]))
print("full coefficients:", np.round(full.coef_, 2))
print("streamed predictions close to full fit:", bool(np.abs(pred_full - pred_sgd).max() < 0.3))

full coefficients: [ 3.  -2.   0.5]
streamed predictions close to full fit: True


## 22.12 How to solve any big-data and fast-dataframe problem: the playbook

In [ ]:
import numpy as np
import pandas as pd

def compare(trusted, candidate, keys, value_cols):
    """Return a list of human-readable differences (empty list means they agree)."""
    problems = []
    if len(trusted) != len(candidate):
        problems.append(f"row count {len(trusted)} vs {len(candidate)}")
    if candidate.duplicated(keys).any():
        problems.append("duplicate keys in candidate")
    merged = trusted.merge(candidate, on=keys, how="outer", suffixes=("_t", "_c"), indicator=True)
    only = merged["_merge"].value_counts()
    if only.get("left_only", 0) or only.get("right_only", 0):
        problems.append(f"keys only in trusted: {only.get('left_only', 0)}, only in candidate: {only.get('right_only', 0)}")
    both = merged[merged["_merge"] == "both"]
    for c in value_cols:
        if not np.allclose(both[c + "_t"], both[c + "_c"]):
            problems.append(f"values differ in {c}")
    return problems

orders = pd.DataFrame({"cid": [1, 2, 3, 4], "amount": [10.0, 20.0, 30.0, 40.0]})
tiers = pd.DataFrame({"cid": [1, 2, 3], "tier": ["g", "s", "g"]})           # customer 4 unknown
trusted = orders.merge(tiers, on="cid", how="left").groupby("cid", as_index=False)["amount"].sum()
buggy = orders.merge(tiers, on="cid", how="inner").groupby("cid", as_index=False)["amount"].sum()
print("good:", compare(trusted, trusted.copy(), ["cid"], ["amount"]))
print("bug :", compare(trusted, buggy, ["cid"], ["amount"]))

good: []
bug : ['row count 4 vs 3', 'keys only in trusted: 1, only in candidate: 0']


## 22.13 Going deeper: what is under the hood, and what interviewers probe

## Chapter recap

## Practice set

Twenty-three problems, easy first. Try each before reading the answer. Code in the problems that is marked as a block was run, and the printed results quoted in the answers are real.

In [ ]:
import io
import pandas as pd

csv = "k,v\na,1\nb,2\na,3\nb,4\na,5"
tot = None
for ch in pd.read_csv(io.StringIO(csv), chunksize=2):
    s = ch.groupby("k")["v"].sum()
    tot = s if tot is None else tot.add(s, fill_value=0)
print(tot.to_dict())

In [ ]:
import numpy as np

keys = np.array([0, 0, 0, 0, 1, 2, 3, 4, 8])
sizes = np.bincount(keys % 4, minlength=4)
print(sizes.tolist(), round(sizes.max() / sizes.mean(), 2))

In [ ]:
def update(state, chunk):
    total, count = state
    return total + sum(chunk), count + len(chunk)

state = (0, 0)
for chunk in ([1, 2, 3], [4, 5]):
    state = update(state, chunk)
print(state, state[0] / state[1])

In [ ]:
a = (1e16 + 1.0) - 1e16
b = (1e16 - 1e16) + 1.0
print(a, b)

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch22.html